<a href="https://colab.research.google.com/github/TManhNguyen/OneClick-PrettyMaps/blob/main/PrettyMaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneClick PrettyMaps 🗺️

Colourful map posters of any place, in four steps. Run the cells top to bottom with the ▶ buttons.

1. **Setup** – installs everything (once per session, ~1–2 min).
2. **Choose the area** – search or click on the map, pick circle / square, or draw your own shape.
3. **Download** – fetches OpenStreetMap data for that area.
4. **Style it** – pick a palette (or paste any [Color Hunt](https://colorhunt.co) link) and effects. Re-run step 4 as often as you like; nothing is downloaded again.

Built on [prettymaps](https://github.com/marceloprates/prettymaps) by **Marcelo Prates** · Map data © [OpenStreetMap](https://www.openstreetmap.org/copyright) contributors · Licensed AGPL-3.0

In [ ]:
#@title 1 · Setup (run once)
Version = "main" #@param {type:"string"}
!pip install -q "oneclick-prettymaps[picker] @ git+https://github.com/TManhNguyen/OneClick-PrettyMaps@{Version}"
!pip install -q --force-reinstall --no-deps "oneclick-prettymaps @ git+https://github.com/TManhNguyen/OneClick-PrettyMaps@{Version}"
# prettymaps goes last and without its own dependency list: that list upgrades
# ipykernel (breaks Colab) and pulls ~400 MB of pen-plotter packages we don't use.
# Installing it last also keeps pip from printing warnings about that list.
!pip install -q --no-deps "prettymaps @ git+https://github.com/marceloprates/prettymaps@02f85870ced807b7d24ce1764764ff877f9edffd"

from google.colab import output, files
output.enable_custom_widget_manager()

import sys
# Forget any copy imported earlier in this runtime, so the version just installed is used
for name in [m for m in sys.modules if m.split(".")[0] == "oneclick_prettymaps"]:
    del sys.modules[name]

import matplotlib.pyplot as plt
# Sharp poster previews on high-resolution screens
%config InlineBackend.figure_format = "retina"
import oneclick_prettymaps as opm

print("Fonts:", opm.load_fonts())
print(f"oneclick_prettymaps {opm.__version__} ready ✔  (needs 0.3.0+)")

In [ ]:
#@title 2 · Choose the area
#@markdown Type a place in the **Find** box above the map (e.g. `Phường Tân Phong, Hồ Chí Minh`).
#@markdown If it has an official boundary (ward, district, park…) it is outlined, the map is centred on it
#@markdown and the radius is set to cover all of it. Then pick a shape:
#@markdown - **Circle / Square**: the area around it (adjust the radius if you like)
#@markdown - **Boundary only**: just that place, neighbouring areas removed
#@markdown - **Drawn shape**: draw any polygon or rectangle yourself
#@markdown
#@markdown You can also click the map to move the pin. If the pin looks stuck in a corner, press **Recentre**.
Start_at = "Ho Chi Minh City" #@param {type:"string"}

picker = opm.MapPicker(Start_at, radius=1000).show()

In [ ]:
#@title 3 · Download map data
#@markdown Downloads the area chosen in step 2. Finish choosing it there first.
#@markdown
#@markdown Sea is slower to compute — tick it for coastal places only.
Include_sea = False #@param {type:"boolean"}
#@markdown ---
#@markdown **This object only** (same as **Boundary only** on the map in step 2, but by name): map just one place
#@markdown cut out along its official boundary, neighbouring areas removed, centred with a small border (step 4).
#@markdown Uses **Object_name**, or **Start_at** from step 2 if empty. E.g. `Phường Tân Hưng, Hồ Chí Minh`,
#@markdown or an OpenStreetMap id like `R1234567` if the name is not found.
This_object_only = False #@param {type:"boolean"}
Object_name = "" #@param {type:"string"}
#@markdown Show the map building up as each layer downloads.
Live_preview = True #@param {type:"boolean"}

if This_object_only:
    place = Object_name.strip() or globals().get("Start_at", "")
    selection = dict(query=place, radius=None, circle=False)
elif "picker" in globals():
    selection = picker.selection()
else:
    raise RuntimeError("Run step 2 first and choose the area on the map.")

gdfs = opm.fetch(**selection, sea=Include_sea, progress=opm.LivePreview() if Live_preview else None)

In [ ]:
#@title 4 · Style it (re-run freely — no re-download)
#@markdown **Palette**: pick one, or paste a Color Hunt link like `https://colorhunt.co/palette/f9ed69f08a5db83b5e6a2c70`
Palette = "saigon-sunset" #@param ["saigon-sunset", "pastel-pop", "tropical", "terracotta", "nordic", "mint-candy", "retro-80s", "autumn", "ink", "neon-night", "blueprint"] {allow-input: true}
Mode = "auto" #@param ["auto", "light", "dark"]
#@markdown **Buildings** – `auto` uses building types when the city has them tagged, otherwise size.
Buildings = "auto" #@param ["auto", "type", "height", "size", "distance", "random"]
#@markdown **Streets** – `orientation` = rainbow by compass direction.
Streets = "type" #@param ["type", "orientation", "solid"]
Glow = "auto" #@param ["auto", "on", "off"]
Shadows = False #@param {type:"boolean"}
Hatch = False #@param {type:"boolean"}
#@markdown **Grain**: paper texture strength. **Seed** changes the grain pattern (and colours when Buildings = random).
Grain = 0.05 #@param {type:"slider", min:0, max:0.3, step:0.01}
#@markdown **Land use** tints residential / commercial / industrial areas — brings colour where few buildings are mapped.
Land_use = 0.35 #@param {type:"slider", min:0, max:0.7, step:0.05}
#@markdown ---
Title = "Sai Gon" #@param {type:"string"}
Subtitle = "auto" #@param {type:"string"}
Legend = False #@param {type:"boolean"}
Layout = "poster" #@param ["poster", "square", "a4", "wallpaper"]
#@markdown Leave **Seed** empty for a random one; the seed used is printed so you can paste it back to keep that look.
Seed = "" #@param {type:"string"}
#@markdown **Border**: empty space around the map shape, % of its width on each side.
Border = 2 #@param {type:"slider", min:0, max:15, step:0.5}
#@markdown ---
#@markdown **For printing, use PDF**: it is vector (no pixels), so it stays sharp at any size and the printer picks the resolution.
Download_PDF = False #@param {type:"boolean"}
Download_SVG = False #@param {type:"boolean"}
#@markdown PNG is pixels: 8×10" poster = 7.2 MP at 300 dpi, 28.8 MP at 600 dpi.
Download_PNG = False #@param {type:"boolean"}
DPI = 300 #@param {type:"slider", min:100, max:600, step:50}

import re

seed = Seed.strip() or opm.random_seed_name()
print(f"Seed: {seed}")

fig = opm.render(
    gdfs,
    theme=Palette,
    dark=None if Mode == "auto" else Mode == "dark",
    glow=None if Glow == "auto" else Glow == "on",
    buildings=Buildings,
    streets=Streets,
    shadows=Shadows,
    hatch=Hatch,
    grain=Grain,
    landuse=Land_use,
    title=Title or None,
    subtitle=Subtitle or None,
    legend=Legend,
    layout=Layout,
    seed=seed,
    margin=Border / 100,
)
plt.show()

name = re.sub(r"\W+", "_", Title or "map").strip("_").lower() or "map"
if Download_PDF:
    files.download(opm.save(fig, f"{name}.pdf"))
if Download_SVG:
    files.download(opm.save(fig, f"{name}.svg"))
if Download_PNG:
    w, h = (fig.get_size_inches() * DPI).astype(int)
    print(f"PNG: {w}×{h} px = {w * h / 1e6:.1f} MP")
    files.download(opm.save(fig, f"{name}.png", dpi=DPI))

In [ ]:
#@title 5 · Palette gallery (optional) — preview every palette on your map
#@markdown Can take a minute for big areas.
Gallery_streets = "orientation" #@param ["type", "orientation", "solid"]

for p in opm.PALETTES:
    f = opm.render(gdfs, theme=p, streets=Gallery_streets, title=p, subtitle=None,
                   layout="square", figsize=(4, 4))
    plt.show()